# Tiled Creative Upscaler → Modular Diffusers — smoke

Cheap correctness check: publish **PRIVATE** `remyxai/tiled-upscaler-flux-modular` → load via
`trust_remote_code` → **×2 on a small image** (one 512-px tile, 4 steps) → assert the block class and that
the output is bigger than the input. Full quantitative validation (resolution · Laplacian sharpness ·
seam check) lives in `e2e.ipynb`. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 2 · Publish PRIVATE (upload block.py first)

In [ ]:
import os
from huggingface_hub import HfApi
for f in ["block.py", "modular_config.json", "modular_model_index.json"]:
    assert os.path.exists(f), f"upload {f} next to this notebook first."
api = HfApi(); REPO = "remyxai/tiled-upscaler-flux-modular"
api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
for f in ["block.py", "modular_config.json", "modular_model_index.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))

## 3 · Load + assert the block class

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained(REPO, trust_remote_code=True)
assert type(pipe.blocks).__name__ == "TiledUpscalerBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect TiledUpscalerBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

## 4 · Smoke — ×2 on a small image

One 512-px tile and 4 steps: exercises upscale → tile → img2img refine → feather-blend → decode without
paying for a full canvas.

In [ ]:
import torch
from PIL import Image
from IPython.display import display
src = Image.new("RGB", (256, 256), (120, 90, 60))   # tiny synthetic input, 1 tile after ×2
for i in range(8):                                   # a little structure so there is detail to refine
    for j in range(8):
        if (i + j) % 2: Image.Image.paste(src, (i * 32, j * 32), Image.new("RGB", (32, 32), (200, 160, 90)))
src.save("smoke_src.png"); print("source:", src.size)

In [ ]:
g = torch.Generator(DEV).manual_seed(0)
out = pipe(image="smoke_src.png", scale=2, tile_size=512, tile_overlap=64,
           denoise_strength=0.4, num_inference_steps=4, guidance_scale=3.5, generator=g).images[0]
out.save("smoke_out.png")
print("[SMOKE] source", (256, 256), "-> output", out.size)
display(out.resize((256, 256)))

## Verdict

`loaded block: TiledUpscalerBlock` + an output **larger than the input** (`(256,256) → (512,512)`) with no
error = the block loads through `trust_remote_code` and the tile→refine→blend path runs. If this passes but
the full-size result looks soft, raise `denoise_strength` toward 0.5. Then run `e2e.ipynb` for the
quantitative checks (resolution, Laplacian sharpness, seam-free blending).